CHECKING EVERYTHING OK - UNDERSTAND PROCESS

In [57]:
import os
import torch
import json
import glob
import shutil
import numpy as np
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm  # Use notebook version of tqdm
from transformers import DetrImageProcessor, DetrForObjectDetection
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import pycocotools.mask as mask_utils

### MOCK DATA GENERATOR

In [58]:
# Paths on your server
SERVER_ROOT = '/home/mcv/datasets/C5/KITTI-MOTS/'
MINI_ROOT = './mini_real_kitti'

def create_mini_dataset(source_root, target_root, num_images=5):
    if os.path.exists(target_root):
        shutil.rmtree(target_root)
    
    # We will use Sequence 0002 (Part of Validation Split)
    seq_id = '0002'
    
    # 1. Create Directories
    target_img_dir = os.path.join(target_root, 'training', 'image_02', seq_id)
    target_lbl_dir = os.path.join(target_root, 'instances_txt')
    os.makedirs(target_img_dir, exist_ok=True)
    os.makedirs(target_lbl_dir, exist_ok=True)
    
    # 2. Copy Annotation File
    # We copy the full text file. The Dataset class will just ignore frames we don't have images for.
    src_txt = os.path.join(source_root, 'instances_txt', f'{seq_id}.txt')
    dst_txt = os.path.join(target_lbl_dir, f'{seq_id}.txt')
    
    if os.path.exists(src_txt):
        shutil.copy(src_txt, dst_txt)
        print(f"Copied annotation file: {dst_txt}")
    else:
        print(f"Error: Source annotation not found at {src_txt}")
        return

    # 3. Copy Images
    src_img_dir = os.path.join(source_root, 'training', 'image_02', seq_id)
    images = sorted(glob.glob(os.path.join(src_img_dir, "*.png")))
    
    if len(images) == 0:
        print(f"Error: No images found in {src_img_dir}")
        return

    print(f"Copying first {num_images} images from {src_img_dir}...")
    for img_path in images[:num_images]:
        shutil.copy(img_path, target_img_dir)
    
    print(f"Mini-dataset created at {target_root}")

# EXECUTE COPY
create_mini_dataset(SERVER_ROOT, MINI_ROOT, num_images=5)

Copied annotation file: ./mini_real_kitti/instances_txt/0002.txt
Copying first 5 images from /home/mcv/datasets/C5/KITTI-MOTS/training/image_02/0002...
Mini-dataset created at ./mini_real_kitti


###  CLASS DEFINITIONS

In [59]:
# from src.dataset import KittiMotsDataset
# from src.metrics import CocoEvaluator
# from src.models.detr_wrapper import DetrWrapper

In [60]:
class KittiMotsDataset(Dataset):
    def __init__(self, root_dir, split='train', transforms=None):
        self.root_dir = root_dir
        self.transforms = transforms
        self.img_dir = os.path.join(root_dir, 'training', 'image_02')
        self.label_dir = os.path.join(root_dir, 'instances_txt')
        
        # Modified for mock data: only use sequence 0002
        if split == 'train':
            self.seq_ids = ['0002'] 
        else: # val
            self.seq_ids = ['0002']

        self.class_map = {1: 3, 2: 1} # KITTI -> COCO
        self.samples = self._load_samples()

    def _load_samples(self):
        samples = []
        for seq in self.seq_ids:
            seq_path = os.path.join(self.img_dir, seq)
            if not os.path.exists(seq_path): continue
            
            anno_path = os.path.join(self.label_dir, f"{seq}.txt")
            annos = self._parse_txt_annotations(anno_path)
            
            # print(annos) # PRINT HERE
            
            frames = sorted([f for f in os.listdir(seq_path) if f.endswith('.png')])
            for frame in frames:
                frame_id = int(frame.replace('.png', ''))
                samples.append({
                    'path': os.path.join(seq_path, frame),
                    'seq': seq,
                    'frame_id': frame_id,
                    'annos': annos.get(frame_id, [])
                })
        # print(samples) # PRINT HERE
        return samples

    def _parse_txt_annotations(self, txt_path):
        annotations = {}
        if not os.path.exists(txt_path): return annotations
        with open(txt_path, 'r') as f:
            for line in f:
                parts = line.strip().split(' ')
                frame_id = int(parts[0])
                class_id = int(parts[2])
                rle = parts[5]
                height = int(parts[3])
                width = int(parts[4])
                
                if class_id not in [1, 2]: continue
                if frame_id not in annotations: annotations[frame_id] = []
                
                rle_obj = {'counts': rle, 'size': [height, width]}
                bbox = mask_utils.toBbox(rle_obj) 
                annotations[frame_id].append({'bbox': bbox, 'label': class_id})
        return annotations

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample['path']).convert("RGB")
        w, h = image.size
        boxes, labels = [], []
        for anno in sample['annos']:
            x, y, w_box, h_box = anno['bbox']
            boxes.append([x, y, x + w_box, y + h_box])
            labels.append(self.class_map.get(anno['label']))

        target = {
            'boxes': torch.as_tensor(boxes, dtype=torch.float32),
            'labels': torch.as_tensor(labels, dtype=torch.int64),
            'image_id': torch.tensor([idx]),
            'orig_size': torch.as_tensor([h, w]),
            'seq': sample['seq']
        }
        # print(target) # PRINT HERE
        if self.transforms: image = self.transforms(image)
        return image, target

# --- METRICS.PY ---
class CocoEvaluator:
    def __init__(self, dataset):
        self.dataset = dataset
        self.coco_gt = self._build_coco_gt(dataset)
        self.results = []

    def _build_coco_gt(self, dataset):
        print("Building COCO Ground Truth object...")
        coco_dict = {
            "images": [], "annotations": [],
            "categories": [{"id": 1, "name": "person"}, {"id": 3, "name": "car"}]
        }
        anno_id = 1
        for i in range(len(dataset)):
            sample = dataset.samples[i]
            img_id = i
            # Important: We must use the image size from the sample or assume standard
            # For this dry run, we assume standard KITTI
            coco_dict["images"].append({
                "id": img_id, "width": 1242, "height": 375, "file_name": sample['path']
            })
            for anno in sample['annos']:
                cat_id = dataset.class_map[anno['label']]
                coco_dict["annotations"].append({
                    "id": anno_id, "image_id": img_id, "category_id": cat_id,
                    "bbox": anno['bbox'], "area": anno['bbox'][2] * anno['bbox'][3], "iscrowd": 0
                })
                anno_id += 1
                
        # print(coco_dict)  # PRINT HERE
        
        coco_gt = COCO()
        coco_gt.dataset = coco_dict
        coco_gt.createIndex()
        return coco_gt

    def summarize(self):
        print(f"Evaluating on {len(self.results)} predictions...")
        if not self.results: 
            print("No predictions found!"); return

        # print(self.results) # PRINT HERE
        
        coco_results = []
        for res in self.results:
            img_id = res['image_id']
            for box, score, label in zip(res['boxes'], res['scores'], res['labels']):
                x1, y1, x2, y2 = box
                coco_results.append({
                    "image_id": img_id, "category_id": label,
                    "bbox": [x1, y1, x2 - x1, y2 - y1], "score": score
                })
        
        if not coco_results: print("No valid predictions."); return
        
        coco_dt = self.coco_gt.loadRes(coco_results)
        coco_eval = COCOeval(self.coco_gt, coco_dt, 'bbox')
        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()

# --- DETR_WRAPPER.PY ---
class DetrWrapper:
    def __init__(self, model_name="facebook/detr-resnet-50", device=None):
        self.device = device if device else ('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Loading DETR model: {model_name} on {self.device}...")
        self.processor = DetrImageProcessor.from_pretrained(model_name)
        self.model = DetrForObjectDetection.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

    def predict(self, images, confidence_threshold=0.5): # Lowered thresh for dry run
        if isinstance(images, tuple): images = list(images)
        elif not isinstance(images, list): images = [images]
        
        inputs = self.processor(images=images, return_tensors="pt")
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model(**inputs)
            
        target_sizes = torch.tensor([img.size[::-1] for img in images]).to(self.device)
        
        processed_results = self.processor.post_process_object_detection(
            outputs, target_sizes=target_sizes, threshold=confidence_threshold
        )
        
        standardized_predictions = []
        for result in processed_results:
            pred_dict = {
                'boxes': result['boxes'].cpu().tolist(),
                'scores': result['scores'].cpu().tolist(),
                'labels': result['labels'].cpu().tolist()
            }
            standardized_predictions.append(pred_dict)
            
        # print(standardized_predictions)  # PRINT HERE
        
        return standardized_predictions

### MAIN EXECUTION

In [61]:
def main_real_test():
    print(f"--- Running on MINI REAL Dataset ---")
    
    # Use the MINI_ROOT we created above
    dataset = KittiMotsDataset(root_dir=MINI_ROOT, split='val')
    dataloader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))
    
    print(f"Dataset Loaded: {len(dataset)} images.")
    if len(dataset) > 0:
        print(f"First image path: {dataset.samples[0]['path']}")
    
    # Load Model
    model = DetrWrapper()
    
    # Init Evaluator
    evaluator = CocoEvaluator(dataset)
    
    # Inference
    print("\nStarting Inference...")
    for batch_idx, (images, targets) in enumerate(tqdm(dataloader)):
        # Use a realistic threshold now (e.g., 0.5 or 0.7) since these are real images
        batch_preds = model.predict(images, confidence_threshold=0.5)
        
        for i, pred in enumerate(batch_preds):
            target_id = targets[i]['image_id'].item()
            pred['image_id'] = target_id
            evaluator.results.append(pred)

    # Eval
    print("\n--- Evaluation Results ---")
    evaluator.summarize()

main_real_test()

--- Running on MINI REAL Dataset ---
Dataset Loaded: 5 images.
First image path: ./mini_real_kitti/training/image_02/0002/000000.png
Loading DETR model: facebook/detr-resnet-50 on cpu...


Loading weights: 100%|██████████| 530/530 [00:00<00:00, 830.88it/s, Materializing param=model.query_position_embeddings.weight]                 
DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building COCO Ground Truth object...
creating index...
index created!

Starting Inference...


100%|██████████| 3/3 [00:02<00:00,  1.33it/s]


--- Evaluation Results ---
Evaluating on 5 predictions...
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.00s).
Accumulating evaluation results...
DONE (t=0.01s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.733
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 1.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.891
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.730
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.800
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.308
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.762
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.762
 Average